In [2]:
import pandas as pd
import geopandas as gpd
import osmnx as ox
import networkx as nx

from pyproj import Transformer
from sklearn.neighbors import BallTree
from pathlib import Path
import requests
import zipfile

In [3]:
DATA_DIR = Path("../data")
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [6]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

url_ubicacion_dataset = "https://datos.madrid.es/dataset/202468-0-intensidad-trafico"

response = requests.get(url_ubicacion_dataset)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

enlaces = []

for a in soup.find_all("a", href=True):
    href = urljoin(url_ubicacion_dataset, a["href"])
    texto = a.get_text(" ", strip=True)

    if "/download/" in href or href.lower().endswith((".csv", ".xlsx", ".zip")):
        enlaces.append({
            "texto": texto,
            "url": href
        })

df_enlaces = pd.DataFrame(enlaces).drop_duplicates()
df_enlaces.head(20)

,texto,url
0,Descarga,https://datos.madrid.es/dataset/202468-0-inten...
1,Descarga,https://datos.madrid.es/dataset/202468-0-inten...
2,Descarga,https://datos.madrid.es/dataset/202468-0-inten...
3,Descarga,https://datos.madrid.es/dataset/202468-0-inten...


In [7]:
df_enlaces[df_enlaces["url"].str.contains(".csv", case=False, na=False)]

,texto,url


In [9]:
url_csv = "https://datos.madrid.es/dataset/202468-0-intensidad-trafico/resource/202468-294-intensidad-trafico/download/202468-294-intensidad-trafico.csv"

df_medidores = pd.read_csv(
    url_csv,
    sep=";",
    encoding="latin1"
)

df_medidores.head()

,tipo_elem,distrito,id,cod_cent,nombre,utm_x,utm_y,longitud,latitud
0,other,1.0,6835,18RA28PM01,18RA28PM01,438764.313318,4.474327e+06,-3.721794,40.417315
1,other,9.0,1012,18RA66PM01,18RA66PM01,438740.943152,4.474610e+06,-3.722097,40.419861
2,URB,10.0,5035,95013,FRUELA N-S,438004.401612,4.473859e+06,-3.730705,40.413040
3,URB,5.0,5579,61068,Potosi E-O - Bolivia-Víctor Andrés Belaunde,442420.642251,4.478696e+06,-3.679095,40.456932
4,URB,5.0,5580,61069,Víctor Andrés Belaunde N-S - Cochabamba-Potosi,442366.670466,4.478601e+06,-3.679723,40.456073


In [10]:
df_medidores.columns

Index(['tipo_elem', 'distrito', 'id', 'cod_cent', 'nombre', 'utm_x', 'utm_y',
       'longitud', 'latitud'],
      dtype='object')

In [12]:
medidores = df_medidores[["id", "nombre", "utm_x", "utm_y", "longitud", "latitud"]].copy()

medidores = medidores.dropna(subset=["longitud", "latitud"])
medidores = medidores.drop_duplicates(subset=["id"])

medidores.head()

,id,nombre,utm_x,utm_y,longitud,latitud
0,6835,18RA28PM01,438764.313318,4.474327e+06,-3.721794,40.417315
1,1012,18RA66PM01,438740.943152,4.474610e+06,-3.722097,40.419861
2,5035,FRUELA N-S,438004.401612,4.473859e+06,-3.730705,40.413040
3,5579,Potosi E-O - Bolivia-Víctor Andrés Belaunde,442420.642251,4.478696e+06,-3.679095,40.456932
4,5580,Víctor Andrés Belaunde N-S - Cochabamba-Potosi,442366.670466,4.478601e+06,-3.679723,40.456073


In [13]:
print("Número de medidores:", len(medidores))
medidores.head()

Número de medidores: 5072


,id,nombre,utm_x,utm_y,longitud,latitud
0,6835,18RA28PM01,438764.313318,4.474327e+06,-3.721794,40.417315
1,1012,18RA66PM01,438740.943152,4.474610e+06,-3.722097,40.419861
2,5035,FRUELA N-S,438004.401612,4.473859e+06,-3.730705,40.413040
3,5579,Potosi E-O - Bolivia-Víctor Andrés Belaunde,442420.642251,4.478696e+06,-3.679095,40.456932
4,5580,Víctor Andrés Belaunde N-S - Cochabamba-Potosi,442366.670466,4.478601e+06,-3.679723,40.456073


In [14]:
medidores.dtypes

id            int64
nombre       object
utm_x       float64
utm_y       float64
longitud    float64
latitud     float64
dtype: object

In [15]:
medidores[["latitud", "longitud"]].describe()

,latitud,longitud
count,5072.000000,5072.000000
mean,40.430447,-3.684001
std,0.039163,0.042728
min,40.332454,-3.836886
25%,40.398978,-3.712553
50%,40.431302,-3.686923
75%,40.460080,-3.656194
max,40.515611,-3.551623


In [16]:
G_osm = ox.graph_from_place(
    "Madrid, Spain",
    network_type="drive",
    simplify=True
)

print("Nodos OSM:", len(G_osm.nodes))
print("Aristas OSM:", len(G_osm.edges))

Nodos OSM: 31453
Aristas OSM: 61857


In [17]:
# Comprobar IDs duplicados
print("IDs duplicados:", medidores["id"].duplicated().sum())

# Comprobar coordenadas nulas
print("Latitud nula:", medidores["latitud"].isna().sum())
print("Longitud nula:", medidores["longitud"].isna().sum())

# Comprobar coordenadas fuera de un rango razonable para Madrid
fuera_madrid = medidores[
    ~(
        (medidores["latitud"].between(40.30, 40.55)) &
        (medidores["longitud"].between(-3.90, -3.50))
    )
]

print("Medidores fuera de rango Madrid:", len(fuera_madrid))
fuera_madrid.head()

IDs duplicados: 0
Latitud nula: 0
Longitud nula: 0
Medidores fuera de rango Madrid: 0


,id,nombre,utm_x,utm_y,longitud,latitud


In [18]:
osm_nodes, distancias = ox.distance.nearest_nodes(
    G_osm,
    X=medidores["longitud"].values,
    Y=medidores["latitud"].values,
    return_dist=True
)

medidores["osm_node"] = osm_nodes
medidores["distancia_osm_node_m"] = distancias

medidores.head()

,id,nombre,utm_x,utm_y,longitud,latitud,osm_node,distancia_osm_node_m
0,6835,18RA28PM01,438764.313318,4.474327e+06,-3.721794,40.417315,32636471,79.621034
1,1012,18RA66PM01,438740.943152,4.474610e+06,-3.722097,40.419861,315259372,54.829432
2,5035,FRUELA N-S,438004.401612,4.473859e+06,-3.730705,40.413040,305399713,15.640852
3,5579,Potosi E-O - Bolivia-Víctor Andrés Belaunde,442420.642251,4.478696e+06,-3.679095,40.456932,1672792326,40.116809
4,5580,Víctor Andrés Belaunde N-S - Cochabamba-Potosi,442366.670466,4.478601e+06,-3.679723,40.456073,119794656,14.979371


In [19]:
medidores["distancia_osm_node_m"].describe()

count    5072.000000
mean       36.635675
std        28.015551
min         0.101869
25%        16.227750
50%        28.117214
75%        50.402014
max       345.242185
Name: distancia_osm_node_m, dtype: float64

In [20]:
medidores.sort_values("distancia_osm_node_m", ascending=False).head(20)

,id,nombre,utm_x,utm_y,longitud,latitud,osm_node,distancia_osm_node_m
1938,5268,(TACTICO)SALIDA POLIGONO N-S,434514.277528,4.468632e+06,-3.771300,40.365688,306400716,345.242185
107,4928,(TACTICO) AV. POBLADOS O-E (GIRO A ERICA),435668.604184,4.470451e+06,-3.757889,40.382164,282940163,228.695575
1394,4960,(TACTICO) ERICA N-S (CENTRO C.I.E.),435691.441140,4.470458e+06,-3.757621,40.382233,282940163,219.565346
1760,11199,Fuerzas Armadas - Ciudad Deportiva O-E - Fuerz...,448191.892131,4.481464e+06,-3.611259,40.482247,1012899036,178.720804
2325,11200,Fuerzas Armadas - Ciudad Deportiva O-E (Vía Se...,448192.910496,4.481435e+06,-3.611244,40.481993,1012899118,178.333987
4888,6876,12XC06PM01,441861.911399,4.471142e+06,-3.684994,40.388851,317771984,169.480943
1745,11191,"Av Fuerzas Armadas, 322 O-E - Av Fuerzas Armad...",447371.205461,4.481464e+06,-3.620941,40.482198,969169634,168.795375
1759,11192,"Av Fuerzas Armadas, 322 O-E (Via Servicio) - A...",447372.223825,4.481436e+06,-3.620927,40.481944,969169634,168.326941
446,9916,SINESIO DELGADO O-E (HOSPITAL CARLOS III-ENTRA...,440954.801598,4.480675e+06,-3.696566,40.474658,26205041,163.715082
445,9915,SINESIO DELGADO E-O (SALIDA TUNEL-HOSPITAL CAR...,440949.748176,4.480680e+06,-3.696627,40.474708,26205041,156.229910


In [21]:
import numpy as np
from sklearn.neighbors import BallTree

# Coordenadas en radianes para distancia haversine
coords = np.radians(medidores[["latitud", "longitud"]].values)

tree = BallTree(coords, metric="haversine")

# Calculamos hasta los 20 vecinos más cercanos para estudiar la distribución
K_ANALISIS = 20

distancias, indices = tree.query(coords, k=K_ANALISIS + 1)

R = 6371000  # radio medio de la Tierra en metros

# Convertimos distancias a metros
distancias_m = distancias * R

In [22]:
resumen_vecinos = pd.DataFrame({
    "vecino_1_m": distancias_m[:, 1],
    "vecino_3_m": distancias_m[:, 3],
    "vecino_5_m": distancias_m[:, 5],
    "vecino_10_m": distancias_m[:, 10],
    "vecino_20_m": distancias_m[:, 20],
})

resumen_vecinos.describe(percentiles=[0.25, 0.5, 0.75, 0.90, 0.95])

,vecino_1_m,vecino_3_m,vecino_5_m,vecino_10_m,vecino_20_m
count,5072.000000,5072.000000,5072.000000,5072.000000,5072.000000
mean,60.299512,134.802462,192.218004,303.847693,479.846961
std,50.277204,83.244571,106.243956,150.109013,302.088483
min,0.000000,10.896282,15.656927,80.867585,195.699433
25%,16.317860,89.649261,130.094414,216.112762,344.834643
50%,51.730719,125.062705,172.480334,274.661836,421.074841
75%,90.709754,164.091985,225.365385,348.903363,527.182705
90%,123.780820,211.121938,299.572190,449.004668,660.410756
95%,148.631821,252.537244,367.262305,544.203109,817.526027
max,559.979456,1626.513574,1656.200417,1741.671511,4377.116647


In [23]:
radio_candidatos_m = np.percentile(distancias_m[:, 5], 90)

print("Radio de candidatos calculado:", radio_candidatos_m, "metros")

Radio de candidatos calculado: 299.5721899933185 metros


In [24]:
# Convertimos radio de metros a radianes
radio_candidatos_rad = radio_candidatos_m / R

indices_radio, distancias_radio = tree.query_radius(
    coords,
    r=radio_candidatos_rad,
    return_distance=True,
    sort_results=True
)

pares_candidatos = []

for i in range(len(medidores)):
    medidor_origen = medidores.iloc[i]
    
    for j, distancia_rad in zip(indices_radio[i], distancias_radio[i]):
        # Saltamos el propio medidor
        if i == j:
            continue
        
        medidor_destino = medidores.iloc[j]
        
        pares_candidatos.append({
            "id_origen": medidor_origen["id"],
            "id_destino": medidor_destino["id"],
            "distancia_directa_m": distancia_rad * R,
            "osm_node_origen": medidor_origen["osm_node"],
            "osm_node_destino": medidor_destino["osm_node"]
        })

df_pares = pd.DataFrame(pares_candidatos)

print("Número de pares candidatos:", len(df_pares))
df_pares.head()

Número de pares candidatos: 60494


,id_origen,id_destino,distancia_directa_m,osm_node_origen,osm_node_destino
0,6835,6833,20.104431,32636471,32636471
1,6835,6836,91.872471,32636471,315261895
2,6835,6837,93.934596,32636471,315264896
3,6835,6827,110.969150,32636471,315261895
4,6835,1042,116.845927,32636471,315265031


In [27]:
candidatos_por_medidor = (
    df_pares
    .groupby("id_origen")
    .size()
    .reset_index(name="num_candidatos")
)

candidatos_por_medidor["num_candidatos"].describe()


count    5060.000000
mean       11.955336
std         6.228532
min         1.000000
25%         7.000000
50%        11.000000
75%        16.000000
max        38.000000
Name: num_candidatos, dtype: float64

In [28]:
candidatos_por_medidor.sort_values("num_candidatos").head(20)

,id_origen,num_candidatos
3186,6768,1
3458,7122,1
331,3706,1
3121,6697,1
3117,6693,1
3113,6689,1
3116,6692,1
1498,4988,1
2792,6352,1
2575,6125,1


In [29]:
# Diccionario: nodo OSM -> lista de medidores asociados a ese nodo
osm_node_to_medidores = (
    medidores
    .groupby("osm_node")["id"]
    .apply(list)
    .to_dict()
)

# Grafo final de medidores
G_medidores = nx.DiGraph()

# Añadimos todos los medidores como nodos, sin eliminar ninguno
for _, row in medidores.iterrows():
    G_medidores.add_node(
        row["id"],
        nombre=row["nombre"],
        latitud=row["latitud"],
        longitud=row["longitud"],
        osm_node=row["osm_node"]
    )

print("Nodos en grafo de medidores:", G_medidores.number_of_nodes())

Nodos en grafo de medidores: 5072


In [30]:
def construir_aristas_por_cutoff(medidores, G_osm, cutoff_m):
    osm_node_to_medidores = (
        medidores
        .groupby("osm_node")["id"]
        .apply(list)
        .to_dict()
    )

    aristas = []

    for _, row in medidores.iterrows():
        id_origen = row["id"]
        osm_origen = row["osm_node"]

        try:
            distancias_red = nx.single_source_dijkstra_path_length(
                G_osm,
                source=osm_origen,
                cutoff=cutoff_m,
                weight="length"
            )
        except (nx.NetworkXNoPath, nx.NodeNotFound):
            continue

        for osm_destino, distancia_red_m in distancias_red.items():
            if osm_destino not in osm_node_to_medidores:
                continue

            for id_destino in osm_node_to_medidores[osm_destino]:
                if id_destino == id_origen:
                    continue

                aristas.append({
                    "id_origen": id_origen,
                    "id_destino": id_destino,
                    "osm_node_origen": osm_origen,
                    "osm_node_destino": osm_destino,
                    "distancia_red_m": distancia_red_m
                })

    return pd.DataFrame(aristas).drop_duplicates()